# Section 5.2: OU process with a fixed non-invariant dictionary

OU process $dX_t=-X_t\,dt+\sqrt2\,dW_t$ with $\rho=N(0,1)$, the dictionary $\{1,\sin x,\cos x,\sin 2x,\cos 2x\}$
(not generator invariant, $\mathcal A\sin x=-x\cos x-\sin x\notin\mathcal V_N$), and horizon $T=1$.
Population matrices use 160 Gauss--Hermite nodes; $\mathcal K^T\cos x=e^{-(1-e^{-2T})/2}\cos(e^{-T}x)$ is exact.

The notebook reproduces

* $\Theta_N$, the eigenvalues of $\mathcal A_N$, the leading operator $\mathcal L_{N,T}=\tfrac12\int_0^Te^{(T-\tau)\mathcal A_N}\mathcal D_Ne^{\tau \mathcal A_N}\,d\tau$
  of Theorem 3.3, see (3.6), and its lower bound (3.7);
* Table 6: time-stepping errors $\|\mathcal Q^n-e^{T\mathcal A_N}\|$ of SDMD-1, SDMD-R, SDMD-2 in the operator norm on $\mathcal V_N$, see (2.3), the normalized
  distance of $((\mathcal Q^{[2]})^n-e^{T\mathcal A_N})/\Delta t$ from $\mathcal L_{N,T}$, and the errors against $\mathcal K^T\cos x$;
* Table 7: errors at $\Delta t=0.05$ with empirical matrices from $m$ i.i.d. samples (100 repetitions, seed 20260507) of the
  three schemes and of the exponential $e^{T\widehat{\mathcal A}}$, against $e^{T\mathcal A_N}$ and against $\mathcal K^T\cos x$, with
  percentile bands and the perturbations of the Galerkin matrices.

Runtime: under one minute.

In [1]:
import math

import numpy as np
import scipy.linalg as sla
from numpy.polynomial.hermite import hermgauss
from scipy.integrate import trapezoid
from scipy.linalg import expm, solve

SEED = 20260507
T = 1.0


def normal_quadrature(n=160, std=1.0):
    """Gauss-Hermite nodes and weights for the Gaussian law N(0, std^2)."""
    nodes, weights = hermgauss(n)
    return math.sqrt(2.0) * std * nodes, weights / math.sqrt(math.pi)


def operator_norm(M, G):
    """Operator norm on V_N with Gram matrix G, i.e. ||R M R^{-1}||_2 with G = R^T R."""
    R = sla.cholesky(0.5 * (G + G.T), lower=False)
    Rinv = sla.solve_triangular(R, np.eye(R.shape[0]), lower=False)
    return float(np.linalg.norm(R @ M @ Rinv, 2))


def trig_dictionary(x):
    """psi = {1, sin x, cos x, sin 2x, cos 2x} and its derivatives up to order four, evaluated at x."""
    psi, d1, d2, d3, d4 = (np.zeros((x.size, 5)) for _ in range(5))
    psi[:, 0] = 1.0
    for col, (k, kind) in enumerate([(1, "sin"), (1, "cos"), (2, "sin"), (2, "cos")], start=1):
        s, c = np.sin(k * x), np.cos(k * x)
        if kind == "sin":
            psi[:, col], d1[:, col], d2[:, col], d3[:, col], d4[:, col] = s, k * c, -k**2 * s, -k**3 * c, k**4 * s
        else:
            psi[:, col], d1[:, col], d2[:, col], d3[:, col], d4[:, col] = c, -k * s, -k**2 * c, k**3 * s, k**4 * c
    return psi, d1, d2, d3, d4


def ou_generator_actions(x):
    """psi, A psi and A^2 psi for the OU generator A f = -x f' + f''."""
    psi, d1, d2, d3, d4 = trig_dictionary(x)
    Apsi = -x[:, None] * d1 + d2
    A2psi = x[:, None] * d1 + (x[:, None] ** 2 - 2.0) * d2 - 2.0 * x[:, None] * d3 + d4
    return psi, Apsi, A2psi


def population(nquad=160):
    """Population matrices under N(0,1): G, H = <psi, A psi>, J = <psi, A^2 psi>, A_N = G^{-1} H, B_N = G^{-1} J."""
    x, w = normal_quadrature(nquad)
    psi, Apsi, A2psi = ou_generator_actions(x)
    cross = lambda L, R: L.T @ (R * w[:, None])
    G, H, J = cross(psi, psi), cross(psi, Apsi), cross(psi, A2psi)
    return x, w, G, H, J, solve(G, H, assume_a="pos"), solve(G, J, assume_a="pos")


def fit_slope(x, y):
    """Least-squares slope of log y against log x."""
    return float(np.polyfit(np.log(np.asarray(x, float)), np.log(np.asarray(y, float)), 1)[0])


def one_step_maps(A, B, dt):
    I = np.eye(A.shape[0])
    return I + dt * A, I + dt * A + 0.5 * dt**2 * (A @ A), I + dt * A + 0.5 * dt**2 * B

## Residual, leading operator, and lower bound

In [2]:
xq, wq, G, H, J, A, B = population()
N = A.shape[0]
I = np.eye(N)
D = B - A @ A
Theta = operator_norm(D, G)

# G-orthonormal representation: A_N is self-adjoint, so its eigenbasis is G-orthonormal
Rch = sla.cholesky(0.5 * (G + G.T), lower=False)
Rinv = sla.solve_triangular(Rch, np.eye(N), lower=False)
As, Ds = Rch @ A @ Rinv, Rch @ D @ Rinv
lam, V = np.linalg.eigh(0.5 * (As + As.T))

ss = np.linspace(0.0, T, 4001)
L = 0.5 * trapezoid(np.array([expm((T - s) * A) @ D @ expm(s * A) for s in ss]), ss, axis=0)
normL = operator_norm(L, G)
lower = 0.5 * T * max(math.exp(T * lam[j]) * float(V[:, j] @ Ds @ V[:, j]) for j in range(N))
ET = expm(T * A)

print(f"Theta_N = {Theta:.4f},  ||A_N|| = {operator_norm(A, G):.4f},  ||B_N|| = {operator_norm(B, G):.4f}")
print("eigenvalues of A_N:", np.round(lam, 4))
print(f"||L_{{N,T}}|| = {normL:.4f};  lower bound (3.7): (T/2) max_j e^(T lam_j) <w_j, D_N w_j> = {lower:.4f}")
print("check of the quadrature: 160 against 200 Gauss-Hermite nodes, max deviation of A_N, B_N:",
      f"{np.abs(A - population(200)[5]).max():.1e}, {np.abs(B - population(200)[6]).max():.1e}")

# observable f = cos x (third dictionary function) and its exact image K^T f
psiq, *_ = trig_dictionary(xq)
f_coef = np.zeros(N); f_coef[2] = 1.0
KTf = math.exp(-0.5 * (1.0 - math.exp(-2.0 * T))) * np.cos(math.exp(-T) * xq)


def l2_err(coef):
    """L^2(rho) distance between sum_j coef_j psi_j and K^T cos x."""
    return float(math.sqrt(np.sum(wq * (psiq @ coef - KTf) ** 2)))


dN = l2_err(ET @ f_coef)
print(f"Galerkin error d_N(T, cos x) = ||e^{{T A_N}} Pi_N cos x - K^T cos x|| = {dN:.4e}")

Theta_N = 3.6599,  ||A_N|| = 5.4399,  ||B_N|| = 32.1726
eigenvalues of A_N: [-5.4399 -4.9254 -2.0892 -1.2014  0.    ]
||L_{N,T}|| = 0.1077;  lower bound (3.7): (T/2) max_j e^(T lam_j) <w_j, D_N w_j> = 0.0815
check of the quadrature: 160 against 200 Gauss-Hermite nodes, max deviation of A_N, B_N: 4.0e-15, 5.7e-14
Galerkin error d_N(T, cos x) = ||e^{T A_N} Pi_N cos x - K^T cos x|| = 9.5512e-03


## Table 6: time-stepping error, leading term, and error against $\mathcal K^T\cos x$

In [3]:
print("   n        dt    SDMD-1     SDMD-R     SDMD-2    rel.Q2  |  SDMD-1(f)  SDMD-R(f)  SDMD-2(f)")
table6 = []
for n in [10, 20, 40, 80, 160, 320]:
    dt = T / n
    Ps = [np.linalg.matrix_power(Q, n) for Q in one_step_maps(A, B, dt)]
    e1, eR, e2 = (operator_norm(P - ET, G) for P in Ps)
    r2 = operator_norm((Ps[2] - ET) / dt - L, G) / normL
    f1, fR, f2 = (l2_err(P @ f_coef) for P in Ps)
    table6.append(dict(n=n, dt=dt, e=(e1, eR, e2), rel=r2, f=(f1, fR, f2)))
    print(f"{n:4d}  {dt:.6f}  {e1:.2e}   {eR:.2e}   {e2:.2e}   {r2:.3f}  |  {f1:.3e}  {fR:.3e}  {f2:.3e}")
dts = [r["dt"] for r in table6]
for j, name in enumerate(["SDMD-1", "SDMD-R", "SDMD-2"]):
    print(f"slope over the three finest steps, {name}: {fit_slope(dts[-3:], [r['e'][j] for r in table6[-3:]]):.2f}")
print("ratio of the time-stepping errors SDMD-2 / SDMD-R:", ", ".join(f"{r['e'][2] / r['e'][1]:.1f}" for r in table6))
print("(SDMD-R(f) - SDMD-2(f)) / d_N:", ", ".join(f"{(r['f'][1] - r['f'][2]) / dN:+.3f}" for r in table6))

   n        dt    SDMD-1     SDMD-R     SDMD-2    rel.Q2  |  SDMD-1(f)  SDMD-R(f)  SDMD-2(f)
  10  0.100000  2.78e-02   2.40e-03   1.62e-02   0.574  |  1.750e-02  9.372e-03  8.974e-03
  20  0.050000  1.37e-02   5.10e-04   6.47e-03   0.224  |  1.262e-02  9.499e-03  9.299e-03
  40  0.025000  6.80e-03   1.22e-04   2.94e-03   0.102  |  1.076e-02  9.538e-03  9.442e-03
  80  0.012500  3.39e-03   3.00e-05   1.41e-03   0.049  |  1.006e-02  9.548e-03  9.501e-03
 160  0.006250  1.69e-03   7.42e-06   6.88e-04   0.024  |  9.778e-03  9.550e-03  9.527e-03
 320  0.003125  8.45e-04   1.85e-06   3.40e-04   0.012  |  9.658e-03  9.551e-03  9.539e-03
slope over the three finest steps, SDMD-1: 1.00
slope over the three finest steps, SDMD-R: 2.01
slope over the three finest steps, SDMD-2: 1.02
ratio of the time-stepping errors SDMD-2 / SDMD-R: 6.8, 12.7, 24.0, 46.9, 92.7, 184.3
(SDMD-R(f) - SDMD-2(f)) / d_N: +0.042, +0.021, +0.010, +0.005, +0.002, +0.001


## Table 7: empirical matrices at $\Delta t=0.05$

For every sample the three schemes and the exponential $e^{T\widehat{\mathcal A}}$ use the same empirical matrices. The errors against
$e^{T\mathcal A_N}$ are errors in the operator norm on $\mathcal V_N$; the errors against $\mathcal K^T\cos x$ are measured in $L^2(\rho)$.

In [4]:
rng = np.random.default_rng(SEED)
dt, n = 0.05, 20
ms = [400, 800, 1600, 3200, 6400, 12800, 25600]
REPS = 100
NAMES = ["exp(T Ahat)", "SDMD-1", "SDMD-R", "SDMD-2"]
floor = dict(zip(NAMES[1:], next(r["e"] for r in table6 if r["n"] == n)))     # time-stepping errors at dt = 0.05
table7 = []
for m in ms:
    errs, ferr, dev = np.zeros((REPS, 4)), np.zeros((REPS, 4)), np.zeros((REPS, 2))
    for r in range(REPS):
        x = rng.standard_normal(m)
        psi, Apsi, A2psi = ou_generator_actions(x)
        Gh, Hh, Jh = psi.T @ psi / m, psi.T @ Apsi / m, psi.T @ A2psi / m
        Ah, Bh = np.linalg.solve(Gh, Hh), np.linalg.solve(Gh, Jh)
        Ps = [expm(T * Ah)] + [np.linalg.matrix_power(Q, n) for Q in one_step_maps(Ah, Bh, dt)]
        errs[r] = [operator_norm(P - ET, G) for P in Ps]
        ferr[r] = [l2_err(P @ f_coef) for P in Ps]
        dev[r] = [operator_norm(Ah - A, G), operator_norm(Bh - B, G)]
    table7.append(dict(m=m, mean=errs.mean(axis=0), lo=np.percentile(errs, 10, axis=0), hi=np.percentile(errs, 90, axis=0),
                       fmean=ferr.mean(axis=0), dev=dev.mean(axis=0),
                       r_beats_2=float(np.mean(errs[:, 2] < errs[:, 3])), r_beats_2_f=float(np.mean(ferr[:, 2] < ferr[:, 3]))))

print(f"errors against e^{{T A_N}} (operator norm on V_N): mean [10th, 90th percentile] over {REPS} repetitions")
print("     m   " + "".join(f"{name:<29s}" for name in NAMES) + "| |Ahat-A|  |Bhat-B|   SDMD-R < SDMD-2")
for t in table7:
    cells = "  ".join(f"{t['mean'][j]:.2e} [{t['lo'][j]:.1e},{t['hi'][j]:.1e}]" for j in range(4))
    print(f"{t['m']:6d}  {cells}  |  {t['dev'][0]:.2e}  {t['dev'][1]:.2e}       {100 * t['r_beats_2']:3.0f}%")
print("\nerrors against K^T cos x (L^2(rho)): means")
print("     m   " + "".join(f"{name:<13s}" for name in NAMES) + "  SDMD-R < SDMD-2   (SDMD-R - SDMD-2) / d_N")
for t in table7:
    print(f"{t['m']:6d}  " + "  ".join(f"{t['fmean'][j]:.3e} " for j in range(4))
          + f"      {100 * t['r_beats_2_f']:3.0f}%            {(t['fmean'][2] - t['fmean'][3]) / dN:+.3f}")
print()
for j, name in enumerate(NAMES):
    print(f"slope in m over the five smallest sample sizes, {name}: {fit_slope(ms[:5], [t['mean'][j] for t in table7[:5]]):.2f}")
print("mean error against e^{T A_N} divided by the time-stepping error of the scheme at dt = 0.05:")
for t in table7:
    print(f"  m = {t['m']:5d}: " + ", ".join(f"{name} {t['mean'][j + 1] / floor[name]:.2f}" for j, name in enumerate(NAMES[1:])))
print("ratio |Bhat - B| / |Ahat - A|:", ", ".join(f"{t['dev'][1] / t['dev'][0]:.1f}" for t in table7))
print("mean error of SDMD-R divided by the mean error of exp(T Ahat), against e^{T A_N}:", ", ".join(f"{t['mean'][2] / t['mean'][0]:.4f}" for t in table7))
print("mean error of SDMD-R divided by the mean error of exp(T Ahat), against K^T cos x:", ", ".join(f"{t['fmean'][2] / t['fmean'][0]:.4f}" for t in table7))
print("mean error against K^T cos x divided by d_N:")
for t in table7:
    print(f"  m = {t['m']:5d}: " + ", ".join(f"{name} {t['fmean'][j] / dN:.3f}" for j, name in enumerate(NAMES)))

errors against e^{T A_N} (operator norm on V_N): mean [10th, 90th percentile] over 100 repetitions
     m   exp(T Ahat)                  SDMD-1                       SDMD-R                       SDMD-2                       | |Ahat-A|  |Bhat-B|   SDMD-R < SDMD-2
   400  4.98e-02 [2.4e-02,8.8e-02]  5.17e-02 [2.5e-02,8.8e-02]  4.98e-02 [2.4e-02,8.8e-02]  4.54e-02 [2.1e-02,8.2e-02]  |  6.60e-01  7.13e+00        16%
   800  3.31e-02 [1.3e-02,5.9e-02]  3.64e-02 [1.8e-02,6.2e-02]  3.31e-02 [1.3e-02,5.9e-02]  3.12e-02 [1.4e-02,5.5e-02]  |  4.47e-01  4.65e+00        28%
  1600  2.25e-02 [9.8e-03,3.9e-02]  2.67e-02 [1.7e-02,4.1e-02]  2.25e-02 [9.7e-03,3.9e-02]  2.17e-02 [9.2e-03,3.8e-02]  |  3.07e-01  3.21e+00        33%
  3200  1.58e-02 [8.0e-03,2.6e-02]  2.09e-02 [1.4e-02,2.9e-02]  1.58e-02 [7.9e-03,2.6e-02]  1.61e-02 [8.0e-03,2.6e-02]  |  2.17e-01  2.30e+00        54%
  6400  1.08e-02 [5.0e-03,1.8e-02]  1.85e-02 [1.3e-02,2.4e-02]  1.08e-02 [4.9e-03,1.8e-02]  1.16e-02 [6.9e-03,1.8e-02]  |  1.

## LaTeX rows of Tables 6 and 7

In [5]:
def sci(x, sig=3):
    mant, ex = f"{x:.{sig-1}e}".split("e")
    return f"${mant}\\times10^{{{int(ex)}}}$"


def unit(x, scale, sig=3):
    """x / scale with sig significant digits in positional notation."""
    y = x / scale
    digits = max(sig - 1 - int(math.floor(math.log10(abs(y)))), 0)
    return f"${round(y, digits):.{digits}f}$"


print("% Table 6 (errors against K^T cos x in units of 1e-3)")
for r in table6:
    print(" & ".join([f"${r['dt']:g}$"] + [sci(e) for e in r["e"]] + [f"${r['rel']:.3f}$"] + [unit(f, 1e-3) for f in r["f"]]) + r" \\")
print("% Table 7, errors against e^{T A_N} in units of 1e-2: mean [10th, 90th percentile]")
for t in table7:
    print(" & ".join([str(t["m"])] + [f"{unit(t['mean'][j], 1e-2)} [{unit(t['lo'][j], 1e-2, 2)}, {unit(t['hi'][j], 1e-2, 2)}]" for j in range(4)]) + r" \\")
print("% Table 7, errors against K^T cos x in units of 1e-2: means")
for t in table7:
    print(" & ".join([str(t["m"])] + [unit(t["fmean"][j], 1e-2) for j in range(4)]) + r" \\")

% Table 6 (errors against K^T cos x in units of 1e-3)
$0.1$ & $2.78\times10^{-2}$ & $2.40\times10^{-3}$ & $1.62\times10^{-2}$ & $0.574$ & $17.5$ & $9.37$ & $8.97$ \\
$0.05$ & $1.37\times10^{-2}$ & $5.10\times10^{-4}$ & $6.47\times10^{-3}$ & $0.224$ & $12.6$ & $9.50$ & $9.30$ \\
$0.025$ & $6.80\times10^{-3}$ & $1.22\times10^{-4}$ & $2.94\times10^{-3}$ & $0.102$ & $10.8$ & $9.54$ & $9.44$ \\
$0.0125$ & $3.39\times10^{-3}$ & $3.00\times10^{-5}$ & $1.41\times10^{-3}$ & $0.049$ & $10.1$ & $9.55$ & $9.50$ \\
$0.00625$ & $1.69\times10^{-3}$ & $7.42\times10^{-6}$ & $6.88\times10^{-4}$ & $0.024$ & $9.78$ & $9.55$ & $9.53$ \\
$0.003125$ & $8.45\times10^{-4}$ & $1.85\times10^{-6}$ & $3.40\times10^{-4}$ & $0.012$ & $9.66$ & $9.55$ & $9.54$ \\
% Table 7, errors against e^{T A_N} in units of 1e-2: mean [10th, 90th percentile]
400 & $4.98$ [$2.4$, $8.8$] & $5.17$ [$2.5$, $8.8$] & $4.98$ [$2.4$, $8.8$] & $4.54$ [$2.1$, $8.2$] \\
800 & $3.31$ [$1.3$, $5.9$] & $3.64$ [$1.8$, $6.2$] & $3.31$ [$1.3$, $5.9

## Software versions of this run

In [6]:
import platform
import scipy

print("Python", platform.python_version(), "| NumPy", np.__version__, "| SciPy", scipy.__version__)

Python 3.13.18 | NumPy 2.5.3 | SciPy 1.18.1
